# Install Requirements

In [4]:
# interact with *OGC API - Processes* using Weaver Python Client
!pip install 'weaver @ git+https://github.com/crim-ca/weaver.git'

# Only for CRIM platform authentication
!pip install 'requests_magpie @ git+https://github.com/Ouranosinc/requests-magpie.git'

# Create *OGC API - Processes* Client and List Processes

> ℹ️ **Note**: <br>
>
> If authentication is not sufficient, the following will fail.

In [5]:
import os
from requests_magpie import MagpieAuth
from weaver.cli import WeaverClient

client = WeaverClient(
    url="https://hirondelle.crim.ca/weaver/",
    auth=MagpieAuth(
        magpie_url="https://hirondelle.crim.ca/magpie/",
        username=os.getenv("OGC_OSPD_USERNAME"),
        password=os.getenv("OGC_OSPD_PASSWORD"),
    )
)

In [4]:
client.processes()

OperationResult(success=True, code=200, message="Listing of available processes successful.")
{
  "$schema": "https://schemas.opengis.net/ogcapi/processes/part1/1.0/openapi/schemas/processList.yaml",
  "description": "Listing of available processes successful.",
  "processes": [
    "EchoProcess",
    "file_index_selector",
    "file2string_array",
    "jsonarray2netcdf",
    "metalink2netcdf",
    "ogc-tb16-land-cover-mapping",
  ],
  "page": 0,
  "count": 6,
  "total": 6,
  "links": [
    {
      "title": "Process listing (no filtering queries applied).",
      "href": "https://hirondelle.crim.ca/weaver/processes",
      "type": "application/json",
      "rel": "collection"
    },
    {
      "title": "Generic query endpoint to list processes.",
      "href": "https://hirondelle.crim.ca/weaver/processes",
      "type": "application/json",
      "rel": "search"
    },
    {
      "title": "Process listing summary (identifiers and count only).",
      "href": "https://hirondelle.crim.c

# Deploy the Algae Use Case Processes

> ℹ️ **Note**: <br>
>
> Weaver pre-checks that step IDs exist in a Workflow. <br>
> Therefore, individual processes must be deployed beforehand. <br>

> ℹ️ **Note**: <br>
>
> This operation does not need to be performed again on [`hirondelle.crim.ca`](https://hirondelle.crim.ca). <br>
> It requires higher access privileges to be completed. <br>
> Processes have been pre-deployed for convenience.  <br>
> This is demonstrated for demonstration purpose only. <br>

> ℹ️ **Note**: <br>
>
> This step assumes that the Docker images built locally have already been pushed to a remote protected registry. <br>
> Docker requirements are adjusted in-place here to leave the original docker references in the CWL definitions, <br>
> such that they can be run locally without access to this remote Docker registry. <br>

In [ ]:
from weaver.utils import load_file

CUR_DIR = os.path.realpath(".")
CWL_DIR = os.path.dirname(CUR_DIR)

for process_id in [
    "calculate-band",
    "download-band-sentinel2-product-safe",
    "download-band-sentinel2-stac-item",
    "select-products-sentinel2",
    "reproject-image",
    "plot-image",
    "algae-usecase-workflow-copernicus-process",
    "algae-usecase-workflow-copernicus",
    "algae-usecase-workflow-earth-search-process",
    "algae-usecase-workflow-earth-search",
]:
    cwl = load_file(os.path.join(CWL_DIR, f"{process_id}.cwl"))
    version = cwl.get("s:softwareVersion", "1.0.0")
    if "requirements" in cwl and "DockerRequirement" in cwl["requirements"]:
        docker_ref = cwl["requirements"]["DockerRequirement"]["dockerPull"]
        docker_img = docker_ref.rsplit("/", 1)[-1]
        if ":" not in docker_ref:
            docker_img = f"{docker_img}:{version}"
        docker_ref = f"docker-registry.crim.ca/ogc-public/ogc-ospd-{docker_img}"
        cwl["requirements"]["DockerRequirement"]["dockerPull"] = docker_ref

    result = client.deploy(
        process_id=process_id,
        cwl=cwl,
        request_timeout=10,
    )
    assert (
        (result.success and result.code == 201) or   # deployment successful
        (not result.success and result.code == 409)  # already deployed
    ), repr(result)


In [6]:
client.processes(detail=False, with_links=False)

OperationResult(success=True, code=200, message="Listing of available processes successful.")
{
  "$schema": "https://schemas.opengis.net/ogcapi/processes/part1/1.0/openapi/schemas/processList.yaml",
  "description": "Listing of available processes successful.",
  "processes": [
    "algae-usecase-workflow-copernicus",
    "algae-usecase-workflow-copernicus-process",
    "calculate-band",
    "download-band-sentinel2-product-safe",
    "download-band-sentinel2-stac-item",
    "EchoProcess",
    "file_index_selector",
    "file2string_array",
    "jsonarray2netcdf",
    "metalink2netcdf",
    "ogc-tb16-land-cover-mapping",
    "plot-image",
    "reproject-image",
    "select-products-sentinel2"
  ],
  "page": 0,
  "count": 14,
  "total": 14
}